In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import itertools
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import filter_demand, load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
from plotly.subplots import make_subplots
# import processing 
# from pivottablejs import pivot_ui
G_save = False

dim = (1000,500)
g_BLUE = "1616A7"
g_GREY = "#7F7F7F"
g_BLUE_SCALE = [
    "rgba(22, 22, 167, 0.08)",
    "rgba(22, 22, 167, 0.13)",
    "rgba(22, 22, 167, 0.18)"
]

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)


In [ ]:
# latex_textwidth_pt = 516.0 # double column # use this command in latex: \the\textwidth


In [ ]:

# day = 7
# ρs = [0.0,0.2,0.3, 0.4,0.5,0.6,0.7,0.8,0.9,0.99]
ss = [
    {'solution_folder': f"RTS-GMLC_v2.1", 'net_demand_type': 'net_load'},
    {'solution_folder': f"RTS-GMLC_v2.1.3", 'net_demand_type': 'load'},
    {'solution_folder': f"RTS-GMLC_v2.1.2", 'net_demand_type': 'wind'},
    {'solution_folder': f"RTS-GMLC_v2.1.1", 'net_demand_type': 'solar'},
    # {'solution_folder': f"RTS-GMLC_v1.1", 'net_demand_type': 'net_load'},
    # {'solution_folder': f"RTS-GMLC_v1.0.5", 'net_demand_type': 'load'},
    # {'solution_folder': f"RTS-GMLC_v1.0.3", 'net_demand_type': 'wind'},
    # {'solution_folder': f"RTS-GMLC_v1.0.6", 'net_demand_type': 'solar'},
    
]
demand= []
random_demand = []
reserve = []
# energy_reserve = []
for s in ss:
    demand_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'uc','Demand.csv'))
    random_demand_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'ed','random_demand.csv'))
    reserve_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'uc','Reserve.csv'))
    demand_  = add_fields(demand_, net_demand_type = s['net_demand_type'])
    random_demand_  = add_fields(random_demand_, net_demand_type = s['net_demand_type'])
    reserve_  = add_fields(reserve_, net_demand_type = s['net_demand_type'])
    # energy_reserve_ = pd.read_csv(os.path.join("..", "input", input_file, 'uc','Energy reserve.csv')) 
    demand.append(demand_)
    random_demand.append(random_demand_)
    reserve.append(reserve_)
    # energy_reserve.append(energy_reserve_)

demand = pd.concat(demand).set_index(['net_demand_type', 'day', 'hour']).sort_index()
random_demand = pd.concat(random_demand).set_index([ 'net_demand_type', 'day', 'hour']).sort_index()
reserve = pd.concat(reserve).set_index(['net_demand_type', 'day', 'hour']).sort_index()

# random_demand = filter_demand(demand, random_demand, reserve)

imbalance = random_demand.sub(demand['demand'], axis=0, level=['net_demand_type', 'day','hour'])


In [ ]:
# # imbalance.stack(future_stack=True).groupby(level=['day', 'hour']).quantile(q=[0.05, 0.25, 0.5, 0.75, 0.95])
# imbalance_quantiles = imbalance.stack(future_stack=True).groupby(level=['day', 'hour', 'net_demand_type']).quantile(q=[0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975])

# imbalance_quantiles.index.set_names(['day', 'hour', 'net_demand_type', 'q'], inplace=True)
# # # Reorder the index to ['day', 'q', 'hour']
# imbalance_quantiles = imbalance_quantiles.reorder_levels(['day', 'q', 'hour', 'net_demand_type']).sort_index()
# imbalance_quantiles

In [ ]:
quantiles =  [[0.025,0.975], [0.05, 0.95], [0.1, 0.9]]
imbalance_quantiles = imbalance.stack(future_stack=True).groupby(level=['net_demand_type', 'day', 'hour',]).quantile(q=sum(quantiles,[]))
imbalance_quantiles.index.set_names(['net_demand_type', 'day', 'hour','q'], inplace=True)
imbalance_mean = imbalance.stack(future_stack=True).groupby(level=['net_demand_type', 'day', 'hour']).mean() 
imbalance_quantiles = imbalance_quantiles.reorder_levels(['net_demand_type', 'day', 'q', 'hour',]).sort_index()
imbalance_quantiles = imbalance_quantiles.reorder_levels(['net_demand_type', 'day', 'q', 'hour']).sort_index()
imbalance_mean = imbalance_mean.sort_index()

In [ ]:
# # After creating your MultiIndex DataFrames, sort them:
# demand = demand.sort_index()
# random_demand = random_demand.sort_index()
# reserve = reserve.sort_index()
# imbalance = imbalance.sort_index()

In [ ]:
# Invert rows and columns: days as rows, net_demand_types as columns

days = [131,132,333]
# net_demand_types = imbalance_quantiles.index.get_level_values('net_demand_type').unique()
net_demand_types = ['net_load']

fig = make_subplots(
    rows=len(net_demand_types), cols =len(days),
    subplot_titles=[f"Day {day}" for day in days] * len(net_demand_types),
    shared_yaxes=True,
    shared_xaxes=True,
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

for col, day in enumerate(days, 1):
    for row, nd_type in enumerate(net_demand_types, 1):
        time = demand.loc[nd_type, day].index.get_level_values('hour') - (day-1)*24
        forecast = demand.loc[nd_type, day]['demand']
        mean = demand.loc[nd_type, day]['demand'] + imbalance_mean.loc[nd_type, day]
        demand_qs = []
        for q in quantiles:
            demand_qs.append([
                demand.loc[nd_type, day]['demand'] + imbalance_quantiles.loc[nd_type, day][q[0]],
                demand.loc[nd_type, day]['demand'] + imbalance_quantiles.loc[nd_type, day][q[1]]
            ])

        blue_iterator = itertools.cycle(g_BLUE_SCALE)
        for (q, d_q) in zip(quantiles, demand_qs):
            fig.add_trace(
                go.Scatter(
                    x=np.concatenate([time, time[::-1]]),
                    y=np.concatenate([d_q[1], d_q[0][::-1]]),
                    fill='toself',
                    fillcolor=next(blue_iterator),
                    line=dict(color='rgba(255,255,255,0)'),
                    hoverinfo="skip",
                    showlegend=(row == 1 and col == 1),
                    name=f'{(q[0]*100)}% - {(q[1]*100)}%'
                ),
                row=row, col=col
            )

        fig.add_trace(
            go.Scatter(
                x=time,
                y=forecast,
                mode='lines+markers',
                name='Forecast',
                line=dict(color='red'),
                showlegend=(row == 1 and col == 1)
            ),
            row=row, col=col
        )
        
        fig.add_trace(
            go.Scatter(
                x=time,
                y=mean,
                mode='lines',
                name='Mean',
                line=dict(color='rgb(29,105,150)'),
                showlegend=(row == 1 and col == 1)
            ),
            row=row, col=col
        )
        
        # Add forecast + reserve_up and forecast - reserve_down
        reserve_up = reserve.loc[nd_type, day]['reserve_up_MW']
        reserve_down = reserve.loc[nd_type, day]['reserve_down_MW']
        
        fig.add_trace(
            go.Scatter(
                x=time,
                y=forecast + reserve_up,
                mode='lines',
                name='Forecast + Reserve Up',
                line=dict(color='green', dash='dash'),
                showlegend=(row == 1 and col == 1)
            ),
            row=row, col=col
        )
        
        fig.add_trace(
            go.Scatter(
                x=time,
                y=forecast - reserve_down,
                mode='lines',
                name='Forecast - Reserve Down',
                line=dict(color='orange', dash='dash'),
                showlegend=(row == 1 and col == 1)
            ),
            row=row, col=col
        )

# Set y-axis titles for each row (day)
for i, nd_type in enumerate(net_demand_types, 1):
    fig.update_yaxes(title_text=f"{nd_type} [MW]", row=i, col=1)
# fig.update_yaxes(title_text=f"Day {day}", row=i, col=1)

fig.update_layout(
    title='Net demand scenarios by day and type',
    template='simple_white',
    height=350 * len(net_demand_types),
    width=350 * len(days),
    # legend=legend_attr,
    legend=dict(
        x=0.5,
        y=-0.35,
        xanchor="center",
        yanchor="bottom",
        orientation="h"
    )
)
fig.show()

In [ ]:

# Invert rows and columns: days as rows, net_demand_types as columns
# Reorder net_demand_types: net_load, load, solar, wind
net_demand_types = ['net_load', 'load', 'solar', 'wind']
# days = [1,131,2]
fig = make_subplots(
    rows=len(days), cols=len(net_demand_types),
    subplot_titles=[f"{nd_type}" for nd_type in net_demand_types] * len(days),
    shared_yaxes=True,
    shared_xaxes=True,
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

for row, day in enumerate(days, 1):
    for col, nd_type in enumerate(net_demand_types, 1):
        time = demand.loc[nd_type, day].index.get_level_values('hour') - (day-1)*24
        # forecast = demand.loc[nd_type, day]['demand']
        mean = imbalance_mean.loc[nd_type, day]
        demand_qs = []
        for q in quantiles:
            demand_qs.append([
                0 + imbalance_quantiles.loc[nd_type, day][q[0]],
                0 + imbalance_quantiles.loc[nd_type, day][q[1]]
            ])

        blue_iterator = itertools.cycle(g_BLUE_SCALE)
        for (q, d_q) in zip(quantiles, demand_qs):
            fig.add_trace(
                go.Scatter(
                    x=np.concatenate([time, time[::-1]]),
                    y=np.concatenate([d_q[1], d_q[0][::-1]]),
                    fill='toself',
                    fillcolor=next(blue_iterator),
                    line=dict(color='rgba(255,255,255,0)'),
                    hoverinfo="skip",
                    showlegend=(row == 1 and col == 1),
                    name=f'{(q[0]*100)}% - {(q[1]*100)}%'
                ),
                row=row, col=col
            )

        fig.add_trace(
            go.Scatter(
                x=time,
                y=mean,
                mode='lines',
                name='Mean',
                line=dict(color='rgb(29,105,150)'),
                showlegend=(row == 1 and col == 1)
            ),
            row=row, col=col
        )
        
        # Add reserve up and reserve down for net_load only
        if nd_type == 'net_load':
            reserve_up = reserve.loc[nd_type, day]['reserve_up_MW']
            reserve_down = reserve.loc[nd_type, day]['reserve_down_MW']
            
            fig.add_trace(
                go.Scatter(
                    x=time,
                    y=reserve_up,
                    mode='lines',
                    name='Reserve Up',
                    line=dict(color='green', dash='dash'),
                    showlegend=(row == 1)
                ),
                row=row, col=col
            )
            
            fig.add_trace(
                go.Scatter(
                    x=time,
                    y=-reserve_down,
                    mode='lines',
                    name='Reserve Down',
                    line=dict(color='orange', dash='dash'),
                    showlegend=(row == 1)
                ),
                row=row, col=col
            )

# Set y-axis titles for each row (day)
for i, day in enumerate(days, 1):
    fig.update_yaxes(title_text=f"Day {day}", row=i, col=1)

fig.update_layout(
    title='Imbalance scenarios by day and net demand type',
    template='simple_white',
    height=200 * len(days),
    width=200 * len(net_demand_types),
    # legend=legend_attr,
    legend=dict(
        x=0.5,
        y=-0.15,
        xanchor="center",
        yanchor="bottom",
        orientation="h"
    )
)
fig.show()

In [ ]:

# Single day plot: day = 131
# Reorder net_demand_types: net_load, load, solar, wind
day = 131
net_demand_types = ['net_load', 'load', 'solar', 'wind']

fig = make_subplots(
    rows=1, cols=len(net_demand_types),
    subplot_titles=[f"{nd_type}" for nd_type in net_demand_types],
    shared_yaxes=True,
    shared_xaxes=True,
    horizontal_spacing=0.05
)

for col, nd_type in enumerate(net_demand_types, 1):
    time = demand.loc[nd_type, day].index.get_level_values('hour') - (day-1)*24
    mean = imbalance_mean.loc[nd_type, day]
    demand_qs = []
    for q in quantiles:
        demand_qs.append([
            0 + imbalance_quantiles.loc[nd_type, day][q[0]],
            0 + imbalance_quantiles.loc[nd_type, day][q[1]]
        ])

    blue_iterator = itertools.cycle(g_BLUE_SCALE)
    for (q, d_q) in zip(quantiles, demand_qs):
        fig.add_trace(
            go.Scatter(
                x=np.concatenate([time, time[::-1]]),
                y=np.concatenate([d_q[1], d_q[0][::-1]]),
                fill='toself',
                fillcolor=next(blue_iterator),
                line=dict(color='rgba(255,255,255,0)'),
                hoverinfo="skip",
                showlegend=(col == 1),
                name=f'{(q[0]*100)}% - {(q[1]*100)}%'
            ),
            row=1, col=col
        )

    fig.add_trace(
        go.Scatter(
            x=time,
            y=mean,
            mode='lines',
            name='Mean',
            line=dict(color='rgb(29,105,150)'),
            showlegend=(col == 1)
        ),
        row=1, col=col
    )
    
    # Add reserve up and reserve down for net_load only
    if nd_type == 'net_load':
        reserve_up = reserve.loc[nd_type, day]['reserve_up_MW']
        reserve_down = reserve.loc[nd_type, day]['reserve_down_MW']
        
        fig.add_trace(
            go.Scatter(
                x=time,
                y=reserve_up,
                mode='lines',
                name='Reserve Up',
                line=dict(color='green'),
                showlegend=True
            ),
            row=1, col=col
        )
        
        fig.add_trace(
            go.Scatter(
                x=time,
                y=-reserve_down,
                mode='lines',
                name='Reserve Down',
                line=dict(color='orange'),
                showlegend=True
            ),
            row=1, col=col
        )

# Set y-axis title
fig.update_yaxes(title_text=f"Imbalance [MW]", row=1, col=1)
fig.update_xaxes(title_text="Hour")
# config = dict(showgrid=False, showticklabels=True, showline=True, linecolor="grey", mirror=True, gridwidth=0.11, gridcolor="grey")

latex_textwidth_pt = 452.0 # single column
scale = 1
dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.7)

fig.update_layout(
    # title=f'Imbalances per component - Day {day}',
    template='simple_white',
    width=dim[0],
    height=dim[1],
    legend=dict(
        x=0.5,
        y=-0.8,
        xanchor="center",
        yanchor="bottom",
        orientation="h"
    )
)

config_x = dict(showgrid=False, showticklabels=True, showline=True, linecolor="grey", mirror=True, gridwidth=0.11, gridcolor="grey")
config_y = dict(showgrid=False, showline=True, linecolor="grey", mirror=True, gridwidth=0.11, gridcolor="grey", title_standoff = 0)
fig.update_xaxes(**config_x)
fig.update_yaxes(**config_y)

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))

fig.show()
if G_save:
    fig.write_image("imbalances_day.pdf", width=dim[0], height=dim[1])
    

In [ ]:
# Calculate correlation between imbalances of 'net_load' and the rest of net_demand_types
# for each day and each hour, across all scenarios (columns)
main_type = 'net_load'
correlations = {}

for other_type in net_demand_types:
    if other_type == main_type:
        continue
    correlations[other_type] = {}
    for day in days:
        correlations[other_type][day] = {}
        for hour in imbalance.loc[main_type, day].index.get_level_values('hour').unique():
            # Get scenario vectors for this (day, hour)
            main_vec = imbalance.loc[main_type, day, hour].values
            other_vec = imbalance.loc[other_type, day, hour].values
            # Compute Pearson correlation (if both have more than 1 scenario)
            if len(main_vec) > 1 and len(other_vec) > 1:
                corr = np.corrcoef(main_vec, other_vec)[0, 1]
            else:
                corr = np.nan
            correlations[other_type][day][hour] = corr

first_l = correlations['load'][days[1]].get(next(iter(correlations['load'][days[1]])))
first_w = correlations['wind'][days[1]].get(next(iter(correlations['wind'][days[1]])))
first_s = correlations['solar'][days[1]].get(next(iter(correlations['solar'][days[1]])))    
# Example: print correlation for 'load' vs 'net_load' for day 1, hour 1
print(f"Correlation between 'net_load' and 'load' for day {days[1]}, hour 1:", first_l)
print(f"Correlation between 'net_load' and 'load' for day {days[1]}, hour 1:", first_w)
print(f"Correlation between 'net_load' and 'load' for day {days[1]}, hour 1:", first_s)
# main_flat = main_imb.values.flatten()
# other_flat = other_imb.values.flatten()
# # Compute Pearson correlation
# corr = np.corrcoef(main_flat, other_flat)[0, 1]
# correlations[other_type] = corr

print("Correlation of imbalances between 'net_load' and other net_demand_types:")
for k, v in correlations.items():
    # Flatten all correlation values into a list
    vals = [corr for day_dict in v.values() for corr in day_dict.values() if not np.isnan(corr)]
    if vals:
        mean_corr = np.mean(vals)
        print(f"{main_type} vs {k}: {mean_corr:.4f}")
    else:
        print(f"{main_type} vs {k}: No valid correlations")

In [ ]:
# Prepare data for plotting: flatten correlations to DataFrame (hour, correlation, type, day)
corr_records = []
for other_type, days_dict in correlations.items():
    for day, hours_dict in days_dict.items():
        for hour, corr in hours_dict.items():
            if not np.isnan(corr):
                corr_records.append({
                    'net_demand_type': other_type,
                    'day': day,
                    'hour': hour,
                    'correlation': corr
                })
corr_df = pd.DataFrame(corr_records)
corr_df.hour += - (corr_df.day - 1) * 24  # Adjust hour to be relative to the first day
# Plot: correlation vs hour, colored by net_demand_type, faceted by day

fig = px.line(
    corr_df,
    x='hour',
    y='correlation',
    color='net_demand_type',
    facet_col ='day',
    # facet_col_wrap=1,
    markers=True,
    title="Hourly Correlation of Imbalances with 'net_load'",
    labels={'correlation': 'Correlation', 'hour': 'Hour'}
)
# fig.update_layout(height=250 * len(days), width=800)
fig.show()

In [ ]:
# Calculate correlation matrices of imbalance per day (hours vs hours)
imbalance_correlation_matrices = {}
unique_days = imbalance.index.get_level_values('day').unique()
for day in unique_days:
    # Select imbalance data for the current day (all scenarios, all hours)
    # imbalance has a MultiIndex with (day, ρ, hour, scenario)
    # We'll unstack to get a DataFrame: rows=scenarios, columns=hours
    day_imbalance = imbalance.loc['net_load',day].reset_index()
    day_imbalance['hour'] = day_imbalance['hour'] - (day - 1) * 24
    day_imbalance.set_index('hour', inplace=True)
    # If imbalance has more than one level (e.g., ρ), stack them into columns
    day_imbalance = day_imbalance.stack().unstack('hour')
    # Compute correlation matrix between hours
    imbalance_correlation_matrices[day] = day_imbalance.corr()
    # first_lag_autocorr[day] = day_imbalance.autocorr(lag=1)


# Example: show correlation matrix for the first day
imbalance_correlation_matrices[unique_days[0]]

In [ ]:


day_to_plot = 5  # or any day from unique_days

# day = 1
# figs = []
subplot_fig = make_subplots(
    rows=1, cols=len(days),
    subplot_titles=[f"Day {d}" for d in days],
    horizontal_spacing=0.05
)

for i, day in enumerate(days):
    z = imbalance_correlation_matrices[day].values
    subplot_fig.add_trace(
        go.Heatmap(
            z=z,
            x=imbalance_correlation_matrices[day].columns,
            y=imbalance_correlation_matrices[day].index,
            colorscale="RdBu",
            zmin=-1, zmax=1,
            colorbar=dict(title="Correlation") if i == len(days)-1 else None,
            showscale=(i == len(days)-1)
        ),
        row=1, col=i+1
    )

subplot_fig.update_layout(
    width=350*len(days),
    height=400,
    title_text="Hourly Correlation Matrices of Imbalances",
    template="simple_white"
)
subplot_fig.show()

In [ ]:
# Select imbalance data for the specified days (all scenarios, all hours)
aux = imbalance.loc[imbalance.index.get_level_values("day").isin(days)] 

# Reset the 'hour' and 'day' indices to columns for easier manipulation
aux.reset_index(['hour','day'], inplace=True)

# Adjust the 'hour' column so that each day starts at hour 1 (subtract offset)
aux['hour'] = aux['hour'] - (aux['day'] - 1) * 24

# Set 'day' and 'hour' as the new index for the DataFrame
aux.set_index(['day','hour'], inplace=True)   

# # Stack any remaining columns (e.g., scenario or ρ), then unstack 'hour' to get a DataFrame with columns as hours
# aux = aux.stack().unstack('hour')
# aux.index.set_names(['day', 'scenario'], inplace=True)
# # Compute first-lag autocorrelation for each hour (column) per (day, scenario)
# first_lag_autocorrelation = aux.apply(lambda row: row.autocorr(lag=1), axis = 1)

first_lag_autocorrelation = aux.groupby(level='day').apply(lambda x: x.apply(lambda col: col.autocorr(lag=1))).stack()


In [ ]:
# Prepare data for plotly boxplot
autocoors_df = first_lag_autocorrelation.reset_index()
autocoors_df.columns = ['day', 'scenario', 'autocorr']

fig = px.box(
    autocoors_df,
    x='day',
    y='autocorr',
    points="all",
    title='First-lag Autocorrelation per Day',
    labels={'autocorr': 'Autocorrelation', 'day': 'Day'}
)

# Add mean line per day
mean_autocorr = autocoors_df.groupby('day')['autocorr'].mean().reset_index()
fig.add_trace(
    go.Scatter(
        x=mean_autocorr['day'],
        y=mean_autocorr['autocorr'],
        mode='markers',
        name='Mean',
        line=dict(color='red', width=2, dash='dash'),
        marker=dict(symbol='circle', size=6)
    )
)

fig.update_layout(width=fig.layout.width // 2 if fig.layout.width else 700, showlegend=True)
fig.show()

